# Indian Small Finance Bank & Private Bank NPA and Capital Audit (No API Key)

This notebook queries the public read-only [LiquiLens](https://liquilens.in/) banking endpoints with zero API key, verifies the response SHA-256 digest, prints reported Gross NPA, Net NPA, CRAR, and CASA disclosures with explicit missingness, and keeps source clocks separate.

- Interactive web screener: https://liquilens.in/banking/compare/
- Filing evidence index: https://liquilens.in/banking/institutions/
- SFB NPA review & CSV: https://liquilens.in/questions/small-finance-banks-rising-npas/

**Boundary:** Public research evidence only. Not a credit rating, deposit-safety verdict, ranking, or investment advice. Missing disclosures stay explicit and are never replaced with zero.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from urllib.request import Request, urlopen

BASE = "https://api.liquilens.in/api/experimental/v1/banking/institutions"
SLUGS = [
    "au-sfb",
    "equitas-sfb",
    "suryoday-sfb",
    "utkarsh-sfb",
    "ujjivan-sfb",
    "federal-bank",
    "cosmos-ucb",
]

def fetch_bank(slug: str) -> dict:
    url = f"{BASE}/{slug}"
    req = Request(url, headers={"Accept": "application/json", "User-Agent": "LiquiLens-Colab-Audit/1.0"})
    with urlopen(req, timeout=20) as resp:
        raw = resp.read()
    payload = json.loads(raw.decode("utf-8"))
    return {
        "url": url,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "data": payload,
    }

rows = []
for slug in SLUGS:
    rec = fetch_bank(slug)
    ev = rec["data"]
    m = ev.get("metrics") or {}
    def val(k):
        item = m.get(k) or {}
        return item.get("value") if item.get("status") == "observed" else None
    rows.append({
        "slug": slug,
        "name": ev.get("name"),
        "sector": ev.get("sector"),
        "period_end": ev.get("period_end"),
        "gnpa_pct": val("gnpa_pct"),
        "nnpa_pct": val("nnpa_pct"),
        "crar_pct": val("crar_pct"),
        "casa_pct": val("casa_pct"),
        "dicgc": (ev.get("deposit_insurance") or {}).get("status"),
        "sha256_prefix": rec["sha256"][:12],
    })

print(f"{'Institution':32s} {'Period':10s} {'GNPA%':>7s} {'NNPA%':>7s} {'CRAR%':>7s} {'CASA%':>7s} {'DICGC':18s}")
print("-" * 96)
for r in rows:
    fmt = lambda x: f"{x:7.2f}" if isinstance(x, (int, float)) else "    N/A"
    print(f"{str(r['name'])[:32]:32s} {str(r['period_end']):10s} {fmt(r['gnpa_pct'])} {fmt(r['nnpa_pct'])} {fmt(r['crar_pct'])} {fmt(r['casa_pct'])} {str(r['dicgc'] or 'unknown'):18s}")

In [ ]:
# Inspect quarter-over-quarter changes where reported (explicitly preserving missingness)
for slug in ["suryoday-sfb", "utkarsh-sfb", "equitas-sfb", "federal-bank"]:
    rec = fetch_bank(slug)
    ev = rec["data"]
    q = (ev.get("changes") or {}).get("quarter") or {}
    print(ev.get("name"), "|", q.get("from_period"), "->", q.get("to_period"), "|", q.get("changes"))